In [1]:
# =============================================================================
#  Hardware-Accessible QGT Proxy — IBM Quantum Task 7
#  Loschmidt Echo Measurement of the Fubini–Study Metric
#
#  PHYSICS AND DERIVATION
#  ────────────────────────────────────────────────────────────────────────────
#  The Fubini–Study metric diagonal component:
#
#    g_μμ = [1 − |⟨Ψ(λ)|Ψ(λ+δ)⟩|²] / δ²   +   O(δ²)
#
#  Hardware measurement via Loschmidt echo:
#    Circuit:  U(λ+δ)† · U(λ) applied to |0⟩^⊗7
#    Measure:  P(0...0) = |⟨0|U†(λ+δ)·U(λ)|0⟩|² = |⟨Ψ(λ)|Ψ(λ+δ)⟩|²
#
#  CIRCUIT SIMPLIFICATION (analytically proved):
#    U(θ,φ) = part2 · Rz(φ)Rx(θ) · part1   (part1,2 independent of θ,φ)
#    U†(θ+δ,φ)·U(θ,φ)
#      = part1† · [Rx(−(θ+δ))Rz(−φ)] · [Rz(φ)Rx(θ)] · part1
#      = part1† · Rx(−δ) · part1
#    The (θ,φ) dependence cancels entirely!
#
#  EXACT ANALYTIC FORMULA (no small-δ approximation):
#    P_00(δ) = |⟨χ|Rx(−δ)|χ⟩|²
#             = 1 − 4 g_θθ sin²(δ/2)
#    where  g_θθ = (1 − ⟨σ_x⟩²_C) / 4  ← from |χ⟩ = part1|0⟩
#
#  CONSEQUENCE: g_θθ is uniform over the entire (θ,φ) parameter space.
#    This analytically explains the Spearman r ≈ 0 between Tr Ω and F_msg.
#
#  HARDWARE STRATEGY:
#    SNR analysis shows: for shots=4000, need δ ≥ 0.5 for SNR > 3.
#    Use δ ∈ {0.5, 0.7, 1.0, 1.3, π/2} and fit P_00(δ) = 1−4g sin²(δ/2)
#    for maximum-likelihood extraction of g_θθ from all δ values jointly.
#
#  ALSO MEASURED:
#    g_φφ(θ): echo for φ-direction — U†(θ,φ+δ)·U(θ,φ) = part1†·Rx(−θ)Rz(−δ)·part1
#    This DOES depend on θ (the Rz generator sees a θ-rotated state).
#    Exact formula: P_00 = 1 − 4g_φφ(θ) sin²(δ/2)
#    with  g_φφ(θ) = (1 − ⟨σ_z⟩²_{C,θ}) / 4
#    where ⟨σ_z⟩_{C,θ} = Tr[σ_z · (Rx(−θ) ρ_C Rx(+θ))]
#
#  Outputs: qgt_echo_results.json
# =============================================================================

import numpy as np
import warnings, json, datetime
warnings.filterwarnings("ignore")
from scipy.optimize import curve_fit
from scipy.stats import beta as beta_dist
from math import pi

from qiskit import QuantumRegister, QuantumCircuit, ClassicalRegister, transpile
from qiskit.quantum_info import (Statevector, DensityMatrix, partial_trace)
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler, Batch

# =============================================================================
#  CONFIG
# =============================================================================
IBM_TOKEN   = "QfkScNfX4bVJ5lXm0082x7F7J6vya3SF5LJZFNOYXqqO"
BACKEND     = "ibm_torino"
SHOTS       = 10000          # high shots — P_00 close to 1 needs precision
N_BOOTSTRAP = 2000
SEED        = 42
RNG         = np.random.default_rng(SEED)
THETA_MSG   = 2.5349076035276403
VARPHI_MSG  = 2.0022404587009195

# δ values chosen for SNR >> 1  (signal 1−P_00 = 4g sin²(δ/2) >> 1/√shots)
DELTAS_THETA = np.array([0.50, 0.70, 1.00, 1.30, pi/2])

# g_φφ(θ) scan: fix δφ=1.0 (good SNR), sweep θ
DELTA_PHI_FIXED = 1.00
THETA_SCAN      = np.linspace(0, pi, 8)

# =============================================================================
#  CIRCUIT BUILDERS
# =============================================================================
def build_part1(theta_msg=THETA_MSG, varphi_msg=VARPHI_MSG):
    """
    Everything BEFORE the V(θ,φ) perturbation in the decoder:
      |χ⟩ = part1|0⟩  (the state that determines g_θθ)
    """
    C=QuantumRegister(1,'C'); E=QuantumRegister(1,'E'); R=QuantumRegister(1,'R')
    A=QuantumRegister(1,'A'); M=QuantumRegister(1,'M'); G=QuantumRegister(1,'G')
    Y=QuantumRegister(1,'Y')
    qc=QuantumCircuit(C,E,R,G,M,A,Y)
    qc.u(theta_msg, varphi_msg, 0.0, M[0])
    qc.swap(C[0], M[0])
    qc.h(E[0]); qc.cx(E[0], M[0])
    qc.h(R[0]); qc.cx(R[0], G[0])
    qc.h(A[0]); qc.cx(A[0], Y[0])
    return qc


def build_echo_theta(delta_theta):
    """
    g_θθ echo:  part1† · Rx(−δ) · part1
    P(0...0) = |⟨χ|Rx(−δ)|χ⟩|² = 1 − 4g_θθ sin²(δ/2)
    Circuit is independent of the base point (θ,φ).
    """
    cr  = ClassicalRegister(7, 'cr')
    p1  = build_part1()
    qc  = QuantumCircuit(*p1.qregs, cr)
    qc  = qc.compose(p1)
    qc.rx(-delta_theta, 0)        # Rx(−δ) on C (qubit 0)
    qc  = qc.compose(p1.inverse())
    qc.measure(range(7), range(7))
    return qc


def build_echo_phi(delta_phi, theta_base):
    """
    g_φφ echo at base angle theta_base:
      part1† · Rx(−θ_base)·Rz(−δ) · part1
    P(0...0) = 1 − 4g_φφ(θ_base) sin²(δ/2)
    """
    cr  = ClassicalRegister(7, 'cr')
    p1  = build_part1()
    qc  = QuantumCircuit(*p1.qregs, cr)
    qc  = qc.compose(p1)
    qc.rz(-delta_phi,    0)       # Rz(−δ) first
    qc.rx(-theta_base,   0)       # then Rx(−θ)
    qc  = qc.compose(p1.inverse())
    qc.measure(range(7), range(7))
    return qc


def build_identity_echo():
    """δ=0 calibration — P(0...0) ≈ 1 − readout_noise."""
    cr = ClassicalRegister(7, 'cr')
    p1 = build_part1()
    qc = QuantumCircuit(*p1.qregs, cr)
    qc = qc.compose(p1).compose(p1.inverse())
    qc.measure(range(7), range(7))
    return qc


# =============================================================================
#  STEP 1 — Analytical predictions
# =============================================================================
print("=" * 65)
print("  Hardware QGT Proxy (Loschmidt Echo) — ibm_torino")
print("=" * 65)

chi       = Statevector.from_instruction(build_part1()).data
rho_C     = partial_trace(DensityMatrix(chi), [1, 2, 3, 4, 5, 6])
sx_C      = float(np.trace(rho_C.data @ np.array([[0,1],[1,0]])).real)
g_tt_exact = (1 - sx_C**2) / 4

print(f"\n  Analytical predictions:")
print(f"    |χ⟩ = part1|0⟩  (fixed, message-dependent)")
print(f"    ⟨σ_x⟩_C = {sx_C:.6f}")
print(f"    g_θθ (exact) = (1 − {sx_C**2:.6f})/4 = {g_tt_exact:.6f}")
print(f"    Exact P_00(δ) = 1 − 4·{g_tt_exact:.4f}·sin²(δ/2)")

print(f"\n  SNR analysis ({SHOTS:,} shots):")
for d in DELTAS_THETA:
    signal = 4 * g_tt_exact * np.sin(d/2)**2
    snr    = signal / (1/np.sqrt(SHOTS))
    print(f"    δ={d:.3f}: 1−P_00={signal:.5f}  SNR={snr:.1f}")

# Simulator P_00 for each δ
def P00_exact_sim(delta, g=g_tt_exact):
    return 1.0 - 4.0 * g * np.sin(delta/2)**2

P00_sim_theta = {float(d): float(P00_exact_sim(d)) for d in DELTAS_THETA}

# Analytical g_φφ(θ)
g_pp_analytical = []
for th in THETA_SCAN:
    cos_t, sin_t = np.cos(th/2), np.sin(th/2)
    Rx_t = np.array([[cos_t, 1j*sin_t], [1j*sin_t, cos_t]])
    rho_C_rot = Rx_t @ rho_C.data @ Rx_t.conj().T
    sz_rot = float(np.trace(rho_C_rot @ np.array([[1,0],[0,-1]])).real)
    g_pp_analytical.append((1 - sz_rot**2) / 4)

print(f"\n  g_φφ(θ) analytical values:")
for th, g in zip(THETA_SCAN, g_pp_analytical):
    print(f"    θ={th:.3f}: g_φφ={g:.6f}")


# =============================================================================
#  STEP 2 — Build, transpile all circuits
# =============================================================================
print(f"\n  Connecting to {BACKEND} ...")
service = QiskitRuntimeService(channel="ibm_quantum_platform", token=IBM_TOKEN)
backend = service.backend(BACKEND)
print(f"  [✓] Connected")

all_circuits = [("identity", build_identity_echo())]
for d in DELTAS_THETA:
    all_circuits.append((f"echo_theta_{d:.4f}", build_echo_theta(d)))
for th in THETA_SCAN:
    all_circuits.append((f"echo_phi_th{th:.4f}", build_echo_phi(DELTA_PHI_FIXED, th)))

print(f"\n  Transpiling {len(all_circuits)} circuits ...")
transpiled = []
for label, qc in all_circuits:
    t = transpile(qc, backend=backend,
                  optimization_level=3, seed_transpiler=SEED)
    transpiled.append((label, t))
    twoq = sum(v for k,v in t.count_ops().items() if k in ['cz','cx','ecr'])
    print(f"    {label:35s}: depth={t.depth():4d}  2q={twoq:3d}")


# =============================================================================
#  STEP 3 — Submit in one Batch
# =============================================================================
print(f"\n  Submitting {len(transpiled)} circuits × {SHOTS} shots ...")
jobs = {}
with Batch(backend=backend) as batch:
    sampler = Sampler(mode=batch)
    sampler.options.dynamical_decoupling.enable = True
    sampler.options.dynamical_decoupling.sequence_type = 'XX'
    for label, t in transpiled:
        jobs[label] = sampler.run([t], shots=SHOTS)


# =============================================================================
#  STEP 4 — Collect results
# =============================================================================
def extract_P00(result):
    pub  = result[0]; data = pub.data
    arr  = data.cr.array.flatten()
    nt   = len(arr)
    nz   = int(sum(1 for b in arr if b == 0))
    return float(nz/nt), nz, nt

def cp_ci(k, n, alpha=0.05):
    lo = beta_dist.ppf(alpha/2,   k, n-k+1) if k > 0 else 0.0
    hi = beta_dist.ppf(1-alpha/2, k+1, n-k) if k < n else 1.0
    return float(lo), float(hi)

print("\n  Collecting results ...")
hw_raw = {}
for label, _ in transpiled:
    print(f"  {label:35s} ... ", end="", flush=True)
    result = jobs[label].result()
    P, nz, nt = extract_P00(result)
    lo, hi    = cp_ci(nz, nt)
    hw_raw[label] = {'P00':P, 'n_zeros':nz, 'n_total':nt, 'P00_lo':lo, 'P00_hi':hi}
    print(f"P00={P:.5f}  [{lo:.5f},{hi:.5f}]")


# =============================================================================
#  STEP 5 — Fit P_00(δ) = 1 − 4g sin²(δ/2) → extract g_θθ
# =============================================================================
print("\n  Fitting P_00(δ) = 1 − 4g·sin²(δ/2) ...")

P_hw_arr  = np.array([hw_raw[f"echo_theta_{d:.4f}"]['P00']    for d in DELTAS_THETA])
P_lo_arr  = np.array([hw_raw[f"echo_theta_{d:.4f}"]['P00_lo'] for d in DELTAS_THETA])
P_hi_arr  = np.array([hw_raw[f"echo_theta_{d:.4f}"]['P00_hi'] for d in DELTAS_THETA])

def model(delta, g):
    return 1.0 - 4.0 * g * np.sin(delta/2)**2

# Weighted nonlinear least-squares fit
sigma_P = (P_hi_arr - P_lo_arr) / (2*1.96)   # approx std from 95% CI
popt, pcov = curve_fit(model, DELTAS_THETA, P_hw_arr,
                        p0=[g_tt_exact], sigma=sigma_P,
                        absolute_sigma=True)
g_tt_fit     = float(popt[0])
g_tt_fit_std = float(np.sqrt(pcov[0,0]))

# Bootstrap on the fit
def bootstrap_g(n=N_BOOTSTRAP):
    gs = []
    for _ in range(n):
        P_boot = np.array([
            float(RNG.beta(hw_raw[f"echo_theta_{d:.4f}"]['n_zeros'] + 1,
                           hw_raw[f"echo_theta_{d:.4f}"]['n_total'] -
                           hw_raw[f"echo_theta_{d:.4f}"]['n_zeros'] + 1))
            for d in DELTAS_THETA
        ])
        try:
            po, _ = curve_fit(model, DELTAS_THETA, P_boot, p0=[g_tt_exact])
            gs.append(float(po[0]))
        except Exception:
            pass
    return np.array(gs)

bs_g = bootstrap_g()
g_tt_bs_lo = float(np.percentile(bs_g, 2.5))
g_tt_bs_hi = float(np.percentile(bs_g, 97.5))

id_P00 = hw_raw['identity']['P00']
print(f"\n  Identity calibration: P_00={id_P00:.5f}  "
      f"(noise floor = {1-id_P00:.5f})")
print(f"\n  g_θθ results:")
print(f"    Fit (weighted NLS):  {g_tt_fit:.6f} ± {g_tt_fit_std:.6f}  (NLS std)")
print(f"    Bootstrap 95% CI:   [{g_tt_bs_lo:.6f}, {g_tt_bs_hi:.6f}]")
print(f"    Analytical:          {g_tt_exact:.6f}")
print(f"    Δg = {g_tt_fit - g_tt_exact:+.6f}  "
      f"({100*abs(g_tt_fit-g_tt_exact)/g_tt_exact:.2f}%)")


# =============================================================================
#  STEP 6 — Extract g_φφ(θ) from θ-scan
# =============================================================================
print(f"\n  g_φφ(θ) scan (δφ={DELTA_PHI_FIXED:.2f}) ...")
g_pp_hw  = []; g_pp_lo = []; g_pp_hi = []

for th, g_anal in zip(THETA_SCAN, g_pp_analytical):
    r = hw_raw[f"echo_phi_th{th:.4f}"]
    P = r['P00']
    # Extract g_φφ from exact formula: P = 1 − 4g sin²(δ/2)
    g = (1 - P) / (4 * np.sin(DELTA_PHI_FIXED/2)**2)
    g_lo = (1 - r['P00_hi']) / (4 * np.sin(DELTA_PHI_FIXED/2)**2)
    g_hi = (1 - r['P00_lo']) / (4 * np.sin(DELTA_PHI_FIXED/2)**2)
    g_pp_hw.append(float(g)); g_pp_lo.append(float(g_lo)); g_pp_hi.append(float(g_hi))
    print(f"  θ={th:.3f}: P00={P:.5f}  g_φφ={g:.5f}  "
          f"analytical={g_anal:.5f}  Δg={g-g_anal:+.5f}")


# =============================================================================
#  STEP 7 — Save
# =============================================================================
class _Enc(json.JSONEncoder):
    def default(self, o):
        if isinstance(o, (np.integer,)): return int(o)
        if isinstance(o, (np.floating,)): return float(o)
        if isinstance(o, np.ndarray):    return o.tolist()
        return super().default(o)

payload = {
    "metadata": {
        "backend": BACKEND, "shots": SHOTS,
        "timestamp": datetime.datetime.utcnow().isoformat() + "Z",
        "theta_msg": THETA_MSG, "varphi_msg": VARPHI_MSG,
        "dd_sequence": "XX",
        "deltas_theta": DELTAS_THETA.tolist(),
        "delta_phi_fixed": float(DELTA_PHI_FIXED),
        "theta_scan": THETA_SCAN.tolist(),
    },
    "analytical": {
        "sx_C":             float(sx_C),
        "g_tt_exact":       float(g_tt_exact),
        "g_pp_analytical":  [float(x) for x in g_pp_analytical],
        "exact_formula":    "P_00(delta) = 1 - 4*g_tt*sin(delta/2)^2",
        "g_tt_formula":     "g_tt = (1 - <sigma_x>_C^2) / 4",
        "note":             "g_tt is uniform over (theta,phi) — analytically proved"
    },
    "simulator": {
        "P00_theta_exact": {str(d): float(P00_exact_sim(d)) for d in DELTAS_THETA},
    },
    "hardware": {
        "identity": {
            "P00": float(id_P00),
            "P00_lo": float(hw_raw['identity']['P00_lo']),
            "P00_hi": float(hw_raw['identity']['P00_hi']),
            "noise_floor": float(1 - id_P00),
        },
        "g_tt": {
            "deltas":       DELTAS_THETA.tolist(),
            "P00_hw":       [float(hw_raw[f"echo_theta_{d:.4f}"]['P00']) for d in DELTAS_THETA],
            "P00_lo":       [float(hw_raw[f"echo_theta_{d:.4f}"]['P00_lo']) for d in DELTAS_THETA],
            "P00_hi":       [float(hw_raw[f"echo_theta_{d:.4f}"]['P00_hi']) for d in DELTAS_THETA],
            "P00_sim":      [float(P00_exact_sim(d)) for d in DELTAS_THETA],
            "g_fit":        float(g_tt_fit),
            "g_fit_std":    float(g_tt_fit_std),
            "g_bs_lo":      float(g_tt_bs_lo),
            "g_bs_hi":      float(g_tt_bs_hi),
            "g_exact":      float(g_tt_exact),
            "delta_g":      float(g_tt_fit - g_tt_exact),
            "bootstrap_g":  bs_g.tolist(),
        },
        "g_pp": {
            "thetas":            THETA_SCAN.tolist(),
            "delta_phi":         float(DELTA_PHI_FIXED),
            "g_pp_hw":           g_pp_hw,
            "g_pp_lo":           g_pp_lo,
            "g_pp_hi":           g_pp_hi,
            "g_pp_analytical":   [float(x) for x in g_pp_analytical],
        },
    },
}

with open("qgt_echo_results.json", "w") as f:
    json.dump(payload, f, indent=2, cls=_Enc)
print("\n[✓] Saved to qgt_echo_results.json")
print("[✓] Run plot_qgt_echo.py for figures.")

qiskit_runtime_service._discover_account:WARNING:2026-03-22 15:18:25,496: Loading account with the given token. A saved account will not be used.


  Hardware QGT Proxy (Loschmidt Echo) — ibm_torino

  Analytical predictions:
    |χ⟩ = part1|0⟩  (fixed, message-dependent)
    ⟨σ_x⟩_C = -0.238426
    g_θθ (exact) = (1 − 0.056847)/4 = 0.235788
    Exact P_00(δ) = 1 − 4·0.2358·sin²(δ/2)

  SNR analysis (10,000 shots):
    δ=0.500: 1−P_00=0.05773  SNR=5.8
    δ=0.700: 1−P_00=0.11089  SNR=11.1
    δ=1.000: 1−P_00=0.21678  SNR=21.7
    δ=1.300: 1−P_00=0.34543  SNR=34.5
    δ=1.571: 1−P_00=0.47158  SNR=47.2

  g_φφ(θ) analytical values:
    θ=0.000: g_φφ=0.081267
    θ=0.449: g_φφ=0.017245
    θ=0.898: g_φφ=0.039716
    θ=1.346: g_φφ=0.131758
    θ=1.795: g_φφ=0.224062
    θ=2.244: g_φφ=0.247121
    θ=2.693: g_φφ=0.183571
    θ=3.142: g_φφ=0.081267

  Connecting to ibm_torino ...


qiskit_runtime_service.__init__:WARNING:2026-03-22 15:18:28,646: Instance was not set at service instantiation. Free and trial plan instances will be prioritized. Based on the following filters: (tags: None, region: us-east, eu-de), and available plans: (open), the available account instances are: CTCs. If you need a specific instance set it explicitly either by using a saved account with a saved default instance or passing it in directly to QiskitRuntimeService().
qiskit_runtime_service.backends:WARNING:2026-03-22 15:18:28,646: Using instance: CTCs, plan: open


  [✓] Connected

  Transpiling 14 circuits ...
    identity                           : depth=   1  2q=  0
    echo_theta_0.5000                  : depth=   6  2q=  0
    echo_theta_0.7000                  : depth=   6  2q=  0
    echo_theta_1.0000                  : depth=   6  2q=  0
    echo_theta_1.3000                  : depth=   6  2q=  0
    echo_theta_1.5708                  : depth=   6  2q=  0
    echo_phi_th0.0000                  : depth=   6  2q=  0
    echo_phi_th0.4488                  : depth=   6  2q=  0
    echo_phi_th0.8976                  : depth=   6  2q=  0
    echo_phi_th1.3464                  : depth=   6  2q=  0
    echo_phi_th1.7952                  : depth=   6  2q=  0
    echo_phi_th2.2440                  : depth=   6  2q=  0
    echo_phi_th2.6928                  : depth=   6  2q=  0
    echo_phi_th3.1416                  : depth=   6  2q=  0

  Submitting 14 circuits × 10000 shots ...

  identity                            ... P00=0.98160  [0.97877,0.98